In [8]:
import pandas as pd

#importing the dataset.
df = pd.read_csv("FOIA_7a_FY2010_FY2019_asof_260630.csv")
print(df.info())

# Pandas inferred some column types incorrectly on the first read.
# BankZip triggered a mixed-type warning, BorrZip came in as int64
# (leading zeros in northeastern ZIPs would be lost), and NaicsCode
# came in as float64 even though it is an industry classification
# code, not a quantity. All three are identifiers, so they are read
# as text. The five date fields listed in the data dictionary arrived
# as plain text, so they are parsed as dates on load.

C:\Users\harsh\AppData\Local\Temp\ipykernel_10716\1984248170.py:3: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("FOIA_7a_FY2010_FY2019_asof_260630.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545751 entries, 0 to 545750
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AsOfDate                    545751 non-null  object 
 1   Program                     545751 non-null  object 
 2   LocationID                  545750 non-null  float64
 3   BorrName                    545721 non-null  object 
 4   BorrStreet                  545751 non-null  object 
 5   BorrCity                    545751 non-null  object 
 6   BorrState                   545751 non-null  object 
 7   BorrZip                     545751 non-null  int64  
 8   BankName                    545750 non-null  object 
 9   BankFDICNumber              506442 non-null  float64
 10  BankNCUANumber              17691 non-null   float64
 11  BankStreet                  545750 non-null  object 
 12  BankCity                    545750 non-null  object 
 13  BankState     

In [9]:
# we are reimporting by changing the dtypes to avoid above warnings.
df = pd.read_csv("FOIA_7a_FY2010_FY2019_asof_260630.csv",
                dtype = {"BorrZip": str, "BankZip": str, "NaicsCode": str},
                parse_dates = ["AsOfDate","ApprovalDate","FirstDisbursementDate","PaidInFullDate","ChargeOffDate"])
print(df.info())

# All data loaded without any errors.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545751 entries, 0 to 545750
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   AsOfDate                    545751 non-null  datetime64[ns]
 1   Program                     545751 non-null  object        
 2   LocationID                  545750 non-null  float64       
 3   BorrName                    545721 non-null  object        
 4   BorrStreet                  545751 non-null  object        
 5   BorrCity                    545751 non-null  object        
 6   BorrState                   545751 non-null  object        
 7   BorrZip                     545751 non-null  object        
 8   BankName                    545750 non-null  object        
 9   BankFDICNumber              506442 non-null  float64       
 10  BankNCUANumber              17691 non-null   float64       
 11  BankStreet                  545750 non-

In [20]:
# Checking null percentage per column.
print("Null Counts of each column :")
null_count = df.isnull().sum()
print(null_count.sort_values(ascending=False))
print("\nNull Percentage of each column :")
null_percent = null_count / len(df) * 100
print(null_percent.sort_values(ascending=False))

Null Counts of each column :
BankNCUANumber                528060
ChargeOffDate                 511608
FranchiseName                 498305
FranchiseCode                 498199
SoldSecMrktInd                417774
PaidInFullDate                151030
FirstDisbursementDate          67051
BankFDICNumber                 39309
BusinessAge                     1824
BorrName                          30
CongressionalDistrict             29
BusinessType                      24
NaicsDescription                   3
NaicsCode                          2
JobsSupported                      2
BankState                          1
BankCity                           1
BankStreet                         1
BankName                           1
LocationID                         1
BankZip                            1
ProjectCounty                      0
RevolverStatus                     0
LoanStatus                         0
CollateralInd                      0
SBADistrictOffice                  0
ProjectSt

In [22]:
# Cheking the LoanStatus value counts
print("Loan Status counts")
print(df["LoanStatus"].value_counts())

Loan Status counts
LoanStatus
P I F     394721
CANCLD     66744
EXEMPT     50089
CHGOFF     34153
COMMIT        44
Name: count, dtype: int64


In [26]:
# Cheking details when loanstatus is chgoff and chargeoffdate is null
print(df[(df["LoanStatus"] == "CHGOFF") & (df["ChargeOffDate"].isnull())])

         AsOfDate Program  LocationID  \
46331  2026-06-30      7A    507814.0   
231862 2026-06-30      7A     40010.0   
255670 2026-06-30      7A     46391.0   
326429 2026-06-30      7A     76551.0   
334828 2026-06-30      7A    317954.0   
343710 2026-06-30      7A     27783.0   
354930 2026-06-30      7A    317954.0   
420007 2026-06-30      7A     29805.0   
478729 2026-06-30      7A     44449.0   
530544 2026-06-30      7A     29805.0   

                                        BorrName                  BorrStreet  \
46331                        L'ATELIER ROUGE INC  270 LAFAYETTE ST SUITE 604   
231862        SOAR Bagel and Coffee Company, LLC       259 East Broad Street   
255670       Breakthrough Chiropractic Care, LLC          3970 Walnut Street   
326429  Gerou Restaurants LLC dba Rosati's Pizza       W290 N4595 TOLBERT LN   
334828                                PHCDC1 LLC         2017 14th Street NW   
343710                 AGrade Construction, Inc.            102 Coug

In [28]:
# Checking whether the label agrees with its supporting fields.
# CHGOFF rows should have a charge off date and amount, and no other
# status should have either.

print(df.groupby('LoanStatus')['GrossChargeOffAmount'].describe())

# Result: 10 of 34,153 CHGOFF rows have no charge off date and an amount of 0.
# Every non-CHGOFF status has an amount of exactly 0,
# so no charge off value appears where it should not. The label is
# consistent apart from those 10 rows, which is 0.03%.

               count           mean            std  min      25%      50%  \
LoanStatus                                                                  
CANCLD       66744.0       0.000000       0.000000  0.0      0.0      0.0   
CHGOFF       34153.0  154316.880814  295056.723861  0.0  21854.4  59987.7   
COMMIT          44.0       0.000000       0.000000  0.0      0.0      0.0   
EXEMPT       50089.0       0.000000       0.000000  0.0      0.0      0.0   
P I F       394721.0       0.000000       0.000000  0.0      0.0      0.0   

                  75%         max  
LoanStatus                         
CANCLD           0.00        0.00  
CHGOFF      151215.17  4706180.93  
COMMIT           0.00        0.00  
EXEMPT           0.00        0.00  
P I F            0.00        0.00  


In [35]:
# Cheking duplicate values.
print("Duplicate Rows Count: ")
print(df.duplicated().sum())
print(df.duplicated(keep=False).sum())
print("\nChecking deplicates for below rows: ")
cols = ["BorrName", "BankName", "ApprovalDate", "GrossApproval", "LoanStatus"]
dup_mask = df.duplicated(subset=cols, keep=False)
print(df[dup_mask][cols].sort_values(by=cols).head(30))

Duplicate Rows Count: 
1136
2022

Checking deplicates for below rows: 
                               BorrName  \
466898   CAPSTONE EYECARE HOLDINGS, LLC   
466999   CAPSTONE EYECARE HOLDINGS, LLC   
467039   CAPSTONE EYECARE HOLDINGS, LLC   
467048   CAPSTONE EYECARE HOLDINGS, LLC   
467116   CAPSTONE EYECARE HOLDINGS, LLC   
415781           100% ALL NATURAL, INC.   
416049           100% ALL NATURAL, INC.   
452447           1061 VICOTRY PLACE LLC   
452539           1061 VICOTRY PLACE LLC   
250714                13200 Kirkham LLC   
250730                13200 Kirkham LLC   
253199               157 OHIO GRILL INC   
253216               157 OHIO GRILL INC   
449604         1773 P & C Holding, LLC.   
449645         1773 P & C Holding, LLC.   
180800   1980 EMERY FAMILY LIVING TRUST   
180801   1980 EMERY FAMILY LIVING TRUST   
432519              1998 third ave corp   
432651              1998 third ave corp   
424576            2 BISD-COMMERCE, INC.   
424787            2 BISD-C

In [37]:
print(df[df["BorrName"].str.contains("CAPSTONE EYECARE", case=False, na=False)].T)

                                                            466898  \
AsOfDate                                       2026-06-30 00:00:00   
Program                                                         7A   
LocationID                                                112722.0   
BorrName                            CAPSTONE EYECARE HOLDINGS, LLC   
BorrStreet                                         510 E. Memorial   
BorrCity                                             OKLAHOMA CITY   
BorrState                                                       OK   
BorrZip                                                      73114   
BankName                                                 BancFirst   
BankFDICNumber                                             27476.0   
BankNCUANumber                                                 NaN   
BankStreet                             100 N Broadway Ave, Ste 200   
BankCity                                             Oklahoma City   
BankState           

In [38]:
df[df.duplicated(keep=False)].head(6).T

,1415,1489,1495,2494,2575,2752
AsOfDate,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00,2026-06-30 00:00:00
Program,7A,7A,7A,7A,7A,7A
LocationID,106925.0,106925.0,106925.0,75939.0,75939.0,434716.0
BorrName,Pantheon Contstruction Corporation,Pantheon Contstruction Corporation,Pantheon Contstruction Corporation,"Westwink Services, Inc.","Westwink Services, Inc.","Lorien Investments, LLC (EPC) (EPC) & PBC Corp..."
BorrStreet,10808 South Riverfront Parkway,10808 South Riverfront Parkway,10808 South Riverfront Parkway,1810 Cofrin Drive,1810 Cofrin Drive,"20100 N. 51st Avenue, Suite B240"
BorrCity,South Jordan,South Jordan,South Jordan,Green Bay,Green Bay,Glendale
BorrState,UT,UT,UT,WI,WI,AZ
BorrZip,84095,84095,84095,54302,54302,85308
BankName,Mountain America FCU,Mountain America FCU,Mountain America FCU,Associated Bank National Association,Associated Bank National Association,"MainStreet Lender 7(a), LLC"
BankFDICNumber,NaN,NaN,NaN,5296.0,5296.0,NaN


In [39]:
print(df[df.duplicated(keep=False)]["LoanStatus"].value_counts())

LoanStatus
CANCLD    1445
P I F      539
EXEMPT      38
Name: count, dtype: int64


In [44]:
# Deleting the deplicate rows.
df = df.drop_duplicates()
print(f"Count after deleting: {df.shape}")

Count after deleting: (544615, 42)


In [45]:
print(df.describe())

                  AsOfDate     LocationID  BankFDICNumber  BankNCUANumber  \
count               544615  544614.000000   505383.000000    17654.000000   
mean   2026-06-30 00:00:00  100874.267766    17373.619641    42266.601224   
min    2026-06-30 00:00:00      20.000000       35.000000       28.000000   
25%    2026-06-30 00:00:00   32987.000000     3890.000000    24563.000000   
50%    2026-06-30 00:00:00   53803.000000     9366.000000    60216.000000   
75%    2026-06-30 00:00:00  102811.000000    26610.000000    67190.000000   
max    2026-06-30 00:00:00  615592.000000    91280.000000    97103.000000   
std                    NaN  124688.481769    17983.417261    25695.571965   

       GrossApproval  SBAGuaranteedApproval                   ApprovalDate  \
count   5.446150e+05           5.446150e+05                         544615   
mean    3.771010e+05           2.798550e+05  2014-12-28 11:35:31.510149120   
min     1.000000e+03           5.000000e+02            2009-10-01 00:00:

In [47]:
print("Checking if TermInMonths columns as any zero")
print(df[df["TermInMonths"] == 0]["LoanStatus"].value_counts())
print("\nChecking if TermInMonths columns as any interest persiod more than 360 months")
print(df[df["TermInMonths"] > 360]["LoanStatus"].value_counts())
print("\nChecking if InitialInterestRate columns as any zero")
print(df[df["InitialInterestRate"] == 0]["LoanStatus"].value_counts())

# we can find that chgoff has 165 rows with zero TermInMonths we will be checking to see what is the reason.
# 222 loans have a term of zero months, which is not a valid loan term. No systematic cause identified across processing method, 
# fiscal year or loan size. Excluded as unusable, representing 0.04% of records

Checking if TermInMonths columns as any zero
LoanStatus
CHGOFF    165
P I F      57
EXEMPT      6
CANCLD      1
Name: count, dtype: int64

Checking if TermInMonths columns as any interest persiod more than 360 months
LoanStatus
EXEMPT    7
CANCLD    3
P I F     3
Name: count, dtype: int64

Checking if InitialInterestRate columns as any zero
LoanStatus
P I F     5
EXEMPT    2
CANCLD    1
Name: count, dtype: int64


In [48]:
# Checking chgoff rows with zero TermInMonths
zero_term = df[(df["TermInMonths"] == 0) & (df["LoanStatus"] == "CHGOFF")]
print(zero_term["ProcessingMethod"].value_counts())
print(zero_term["ApprovalFY"].value_counts())
print(zero_term["RevolverStatus"].value_counts())
print(zero_term["GrossApproval"].describe())

ProcessingMethod
SBA Express Program                    116
Preferred Lenders Program               20
Patriot Express Loans                    5
Community Express                        4
Small Loan Advantage Initiative          4
7a General                               3
Community Advantage Initiative           3
7a with EWCP                             2
Gulf Opportunity Pilot Loan Program      2
Contract CAPLine                         2
Rural Loan Initiative                    2
Working Capital CAPLine                  1
Export Express                           1
Name: count, dtype: int64
ApprovalFY
2011    27
2015    22
2012    21
2016    20
2018    17
2017    16
2010    14
2013    12
2014    11
2019     5
Name: count, dtype: int64
RevolverStatus
N    96
Y    69
Name: count, dtype: int64
count    1.650000e+02
mean     1.429824e+05
std      3.342858e+05
min      5.000000e+03
25%      2.500000e+04
50%      5.000000e+04
75%      1.200000e+05
max      3.500000e+06
Name: GrossApprova

In [50]:
# Checking guranteed ratio
ratio = df["SBAGuaranteedApproval"]/df["GrossApproval"]
print(ratio.describe())
print(ratio.round(2).value_counts().head(10))

count    544615.000000
mean          0.649652
std           0.151210
min           0.100000
25%           0.500000
50%           0.750000
75%           0.750000
max           1.000000
dtype: float64
0.50    263785
0.75    181618
0.85     64603
0.90     33182
0.80        63
0.60        59
0.70        59
0.74        54
0.83        53
0.73        50
Name: count, dtype: int64


In [52]:
# Dropping columns which are not required.
drop_cols = [
    "BorrName", "BorrStreet", "BorrCity", "BorrZip",   # PII, not predictive
    "PaidInFullDate", "ChargeOffDate", "GrossChargeOffAmount",  # leakage, only known after outcome
    "AsOfDate",           # constant, snapshot date
    "Program",            # constant, all 7A
    "FranchiseName",      # redundant with FranchiseCode
    "NaicsDescription"    # redundant with NaicsCode
]
df = df.drop(columns=drop_cols)
print(df.shape)

(544615, 31)


In [53]:
# Cancelled, undisbursed and exempt loans were removed because they have no final outcome. 
# Exempt loans are live loans whose status is withheld under FOIA Exemption 4. 
# Final dataset is 428,361 rows and 31 columns with a 7.93% charge-off rate.
df = df[df["LoanStatus"].isin(["P I F", "CHGOFF"])]
df = df[df["TermInMonths"] > 0]
print(df.shape)
print(df["LoanStatus"].value_counts())
print(df["LoanStatus"].value_counts(normalize=True) * 100)

(428361, 31)
LoanStatus
P I F     394373
CHGOFF     33988
Name: count, dtype: int64
LoanStatus
P I F     92.065571
CHGOFF     7.934429
Name: proportion, dtype: float64


In [54]:
# Data Validation Findings

# 1. Data types
# After loading the dataset pandas as loaded few columns dtypes incorrectly.
# Two ZIP code columns and NaicsCode were read as numbers, but they are codes, not quantities, 
# so I read them as text. Five date columns came in as plain text and I converted them to dates
# All the incorrect column dtypes were reset and got reimported again.

# 2. Duplicates
# Found 1136 duplicate rows which were identical across all 42 columns.totaling 2022 rows and 886 groups.
# This represents 0.2% of the dataset.None of the duplicate have any chargeoff so removing them does not effect the target distribution.
# SBA issues parallel loans to one borrower on the same day, distinguished only by disbursement date, and where that date is blank the rows collapse.
# All duplicates were removed successfully.

# 3. Missing values
# Missingness is structural rather than defective.Means the missingness is due to a reason. The column PaidInFullDate exists only for paid loans.
# Franchise code is empty for most rows because most businesses are not franchises. Credit union numbers are empty because most lenders are banks.

# 4. Label Check
# The column LoanStatus has 5 different categories. In which 10 charged off loans had no charge off date and amount.
# These makes up of 0.03%(10 of 34153 chargeoff). No loan that was paid in full had a charge off amount, so the label agrees with the other columns.

# 5. Wrong values
# When checking TermInMonths column having any zero. I found 222 rows havinng zero as term in month. This should not happen.
# I checked whether they had anything in common and found nothing, so I removed them. 
# 13 loans had terms over 30 years and 8 had 0% interest, both very small numbers.

# 6. Consistency check
# I divided the guaranteed amount by the loan amount. 99.7% matched the official SBA rules of 50%, 75%, 85% and 90%. 
# No guarantee was bigger than the loan.

# 7. Leakage
# Columns like Paid in full date, charge off date and charge off amount only exist after the loan ends
# These will give away the answer to the model. Hence they are removed from dataset.

# 8. Privacy and licence
# Columns that contains Borrower details like Borrower name and address identify real businesses and are not useful for prediction, 
# so I removed them. The data is US Government Works, so it is free to publish and deploy.

# 9. Columns Removed
# "BorrName", "BorrStreet", "BorrCity", "BorrZip",            # PII, not predictive
# "PaidInFullDate", "ChargeOffDate", "GrossChargeOffAmount",  # leakage, only known after outcome
# "AsOfDate",                                                 # constant, snapshot date
# "Program",                                                  # constant, all 7A
# "FranchiseName",                                            # redundant with FranchiseCode
# "NaicsDescription"                                          # redundant with NaicsCode
# Cancelled, undisbursed and exempt loans were removed because they have no final outcome. 
# Exempt loans are live loans whose status is withheld under FOIA Exemption 4. 
# Final dataset is 428,361 rows and 31 columns with a 7.93% charge-off rate.

# Data validation ends with 428,361 rows, 31 columns, 7.93% charge-off rate. This will be carried to EDA.

In [57]:
# saving the file.
df.to_parquet("sba_7a_validated.parquet", index=False)
check = pd.read_parquet("sba_7a_validated.parquet")
print(check.shape)
print(check.dtypes)

(428361, 31)
LocationID                           float64
BorrState                             object
BankName                              object
BankFDICNumber                       float64
BankNCUANumber                       float64
BankStreet                            object
BankCity                              object
BankState                             object
BankZip                               object
GrossApproval                        float64
SBAGuaranteedApproval                float64
ApprovalDate                  datetime64[ns]
ApprovalFY                             int64
FirstDisbursementDate         datetime64[ns]
ProcessingMethod                      object
InitialInterestRate                  float64
FixedorVariableInterestInd            object
TermInMonths                         float64
NaicsCode                             object
FranchiseCode                         object
ProjectCounty                         object
ProjectState                          obje